In [ ]:
# Imports y configuración para visualización
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.manifold import TSNE
try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

# Configuración de matplotlib
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Asegurar path root
CANDIDATE_ROOT = Path.cwd().resolve()
if CANDIDATE_ROOT.name == "notebooks" and CANDIDATE_ROOT.parent.name == "validation":
    ROOT = CANDIDATE_ROOT.parent.parent
else:
    ROOT = next(
        (p for p in [CANDIDATE_ROOT, *CANDIDATE_ROOT.parents] if (p / "validation" / "clustering.py").exists()),
        CANDIDATE_ROOT,
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Cargar módulos necesarios
import importlib.util

def _load_local_module(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    assert spec is not None and spec.loader is not None
    spec.loader.exec_module(module)
    return module

# Carga de módulos necesarios
clustering_module = _load_local_module("tfg_validation_clustering", ROOT / "validation" / "clustering.py")
datasets_module = _load_local_module("tfg_validation_datasets", ROOT / "validation" / "datasets.py")

GenericKMeans = clustering_module.GenericKMeans
DatasetLoader = datasets_module.DatasetLoader

# Carga de métricas para si se necesita evaluate_fuzzy_clustering
try:
    metrics_module = _load_local_module("tfg_validation_metrics", ROOT / "validation" / "metrics" / "metrics.py")
    evaluate_fuzzy_clustering = metrics_module.evaluate_fuzzy_clustering
    evaluate_hard_clustering = metrics_module.evaluate_hard_clustering
except Exception as e:
    print(f"Warning: Could not load metrics module: {e}")
    evaluate_fuzzy_clustering = None
    evaluate_hard_clustering = None

# Configuración de rutas
RESULTS_DIR = ROOT / "validation" / "results" / "clustering_experiment"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
ARTIFACTS_DIR = RESULTS_DIR / "artifacts"
LOGS_DIR = RESULTS_DIR / "logs"

print(f"Resultados en: {RESULTS_DIR}")
print(f"UMAP disponible: {HAS_UMAP}")

In [ ]:
# Cargar resultados guardados del experimento

# Cargar raw results
raw_results_path = TABLES_DIR / "raw_results.csv"
if raw_results_path.exists():
    df_results = pd.read_csv(raw_results_path)
    print(f"Raw results loaded: {len(df_results)} experiments")
else:
    print(f"Warning: Raw results not found at {raw_results_path}")
    df_results = None

# Cargar summary
summary_path = TABLES_DIR / "summary_by_condition.csv"
if summary_path.exists():
    df_summary = pd.read_csv(summary_path)
    print(f"Summary loaded: {len(df_summary)} conditions")
else:
    print(f"Warning: Summary not found at {summary_path}")
    df_summary = None

# Cargar ranked results
ranked_path = TABLES_DIR / "summary_ranked_full.csv"
if ranked_path.exists():
    df_rank = pd.read_csv(ranked_path)
    print(f"Ranked results loaded")
else:
    df_rank = None

# Cargar best per condition
best_per_cond_path = TABLES_DIR / "summary_best_per_condition.csv"
if best_per_cond_path.exists():
    df_best_per_condition = pd.read_csv(best_per_cond_path)
    print(f"Best per condition loaded: {len(df_best_per_condition)} configurations")
else:
    df_best_per_condition = None

# Cargar global best
best_global_path = TABLES_DIR / "summary_best_global_top20.csv"
if best_global_path.exists():
    df_best_global = pd.read_csv(best_global_path)
    print(f"Global best loaded: {len(df_best_global)} configurations")
else:
    df_best_global = None

print("\nResultados cargados exitosamente.")

In [ ]:
# Cargar datasets (para acceso a etiquetas reales)
loader = DatasetLoader()

df_dialogsum = loader.load_dialogsum()
df_stack = loader.load_stackoverflow()
df_esquad = loader.load_esquad()

for df, name in [
    (df_dialogsum, "dialogsum"),
    (df_stack, "stackoverflow"),
    (df_esquad, "esquad"),
]:
    df["dataset"] = name

datasets = {
    "dialogsum": df_dialogsum[["text", "label", "dataset"]].copy(),
    "stackoverflow": df_stack[["text", "label", "dataset"]].copy(),
    "esquad": df_esquad[["text", "label", "dataset"]].copy(),
}

print("Datasets cargados:")
for name, df in datasets.items():
    print(f"  {name}: {len(df)} documents")

# Cargar representaciones desde cache
representations: dict[str, dict[str, np.ndarray]] = {}
print("\nCargando representaciones desde cache...")

for ds_name in datasets.keys():
    representations[ds_name] = {}
    
    # Load BoW
    bow_cache = ARTIFACTS_DIR / f"{ds_name}_bow.pkl"
    if bow_cache.exists():
        with open(bow_cache, "rb") as f:
            payload = pickle.load(f)
        representations[ds_name]["bow"] = payload["X"]
        print(f"  {ds_name}/bow loaded")
    else:
        print(f"  Warning: {ds_name}/bow not found")
    
    # Load TF-IDF
    tfidf_cache = ARTIFACTS_DIR / f"{ds_name}_tfidf.pkl"
    if tfidf_cache.exists():
        with open(tfidf_cache, "rb") as f:
            payload = pickle.load(f)
        representations[ds_name]["tfidf"] = payload["X"]
        print(f"  {ds_name}/tfidf loaded")
    else:
        print(f"  Warning: {ds_name}/tfidf not found")
    
    # Load Nomic
    nomic_cache = ARTIFACTS_DIR / f"{ds_name}_nomic.npy"
    if nomic_cache.exists():
        representations[ds_name]["nomic"] = np.load(nomic_cache)
        print(f"  {ds_name}/nomic loaded")
    else:
        print(f"  Warning: {ds_name}/nomic not found")

print("\nRepresentaciones cargadas:", {ds: list(representations[ds].keys()) for ds in representations})

In [15]:
# Curvas de métrica vs k
curve_metrics = ["asw_mean", "ch_mean", "xb_mean", "pc_mean", "pe_mean"]

for (ds, rep), grp in df_summary.groupby(["dataset", "representation"]):
    fig, axes = plt.subplots(1, len(curve_metrics), figsize=(5 * len(curve_metrics), 4), sharex=True)
    for ax, metric in zip(axes, curve_metrics):
        plot_df = grp.copy()
        plot_df["variant"] = plot_df["algorithm"] + "_" + plot_df["distance"]
        sns.lineplot(data=plot_df, x="k", y=metric, hue="variant", marker="o", ax=ax)
        ax.set_title(f"{metric} vs k")
        ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"Metricas vs k | {ds} | {rep}", y=1.02)
    save_figure(fig, f"curves_{ds}_{rep}.png")
    plt.close(fig)

print("Curvas guardadas.")

Curvas guardadas.


In [16]:
# Boxplots de estabilidad entre semillas
df_box = df_results.copy()
df_box["variant"] = df_box["algorithm"] + "_" + df_box["distance"]

for (ds, rep), grp in df_box.groupby(["dataset", "representation"]):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(data=grp, x="variant", y="asw", ax=axes[0])
    axes[0].set_title(f"ASW estabilidad | {ds} | {rep}")
    axes[0].tick_params(axis="x", rotation=30)

    sns.boxplot(data=grp, x="variant", y="ch", ax=axes[1])
    axes[1].set_title(f"CH estabilidad | {ds} | {rep}")
    axes[1].tick_params(axis="x", rotation=30)

    save_figure(fig, f"boxplot_stability_{ds}_{rep}.png")
    plt.close(fig)

print("Boxplots guardados.")

# Heatmaps por dataset: calidad (ASW) y coste (runtime)
for ds, grp_ds in df_summary.groupby("dataset"):
    tmp = grp_ds.copy()
    tmp["algorithm_distance"] = tmp["algorithm"] + "_" + tmp["distance"]

    asw_heat = tmp.groupby(["algorithm_distance", "representation"], as_index=False)["asw_mean"].mean()
    asw_pivot = asw_heat.pivot(index="algorithm_distance", columns="representation", values="asw_mean")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.heatmap(asw_pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"ASW medio por variante | {ds}")
    save_figure(fig, f"heatmap_asw_{ds}.png")
    plt.close(fig)

    runtime_heat = tmp.groupby(["algorithm_distance", "representation"], as_index=False)["runtime_sec_mean"].mean()
    runtime_pivot = runtime_heat.pivot(index="algorithm_distance", columns="representation", values="runtime_sec_mean")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.heatmap(runtime_pivot, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax)
    ax.set_title(f"Runtime medio (s) por variante | {ds}")
    save_figure(fig, f"heatmap_runtime_{ds}.png")
    plt.close(fig)

# Scatter global calidad-coste
df_scatter = df_summary.copy()
df_scatter["algorithm_distance"] = df_scatter["algorithm"] + "_" + df_scatter["distance"]
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=df_scatter,
    x="runtime_sec_mean",
    y="asw_mean",
    hue="representation",
    style="algorithm_distance",
    s=90,
    alpha=0.85,
    ax=ax,
)
ax.set_title("Trade-off global calidad-coste (ASW vs runtime)")
ax.set_xlabel("Runtime medio (s)")
ax.set_ylabel("ASW medio")
ax.legend(loc="best", fontsize=8)
save_figure(fig, "scatter_quality_vs_cost.png")
plt.close(fig)

print("Heatmaps y scatter calidad-coste guardados.")

Boxplots guardados.
Heatmaps y scatter calidad-coste guardados.


In [18]:
# UMAP 2D (o PCA fallback)
from sklearn.decomposition import PCA

def reduce_to_2d(X, method: str = "umap", random_state: int = 42):
    X_arr = X.toarray() if hasattr(X, "toarray") else X
    if method == "umap" and HAS_UMAP:
        reducer = umap.UMAP(n_components=2, random_state=random_state, metric="cosine")
        return reducer.fit_transform(X_arr)
    pca = PCA(n_components=2, random_state=random_state)
    return pca.fit_transform(X_arr)

df_best = df_best_per_condition.copy()

for _, row in df_best.iterrows():
    ds, rep = row["dataset"], row["representation"]
    X = representations[ds][rep]
    if X is None:
        continue

    model = GenericKMeans(
        n_clusters=int(row["k"]),
        algorithm=row["algorithm"],
        distance=row["distance"],
        random_state=RANDOM_SEED,
        m=2.0 if row["algorithm"] == "fcm" else None
    )
    X_arr = X.toarray() if hasattr(X, "toarray") else X
    labels_pred = model.fit_predict(X_arr)

    X2 = reduce_to_2d(X, method="umap", random_state=RANDOM_SEED)
    labels_true = datasets[ds]["label"].astype(str).values

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(X2[:, 0], X2[:, 1], c=labels_pred, s=10, cmap="tab20")
    axes[0].set_title(f"UMAP/PCA por cluster predicho | {ds}-{rep}")

    # Mapea etiquetas reales a enteros para colorear
    _, true_ids = np.unique(labels_true, return_inverse=True)
    axes[1].scatter(X2[:, 0], X2[:, 1], c=true_ids, s=10, cmap="tab20")
    axes[1].set_title(f"UMAP/PCA por etiqueta real | {ds}-{rep}")

    save_figure(fig, f"umap_{ds}_{rep}.png")
    plt.close(fig)

print("Figuras UMAP/PCA guardadas.")

2026-04-15 11:35:09,601 | INFO | tfg_validation_clustering | Fitting K-Means with k=2, distance=cosine


2026-04-15 11:35:09,776 | INFO | tfg_validation_clustering | K-Means converged at iteration 12
2026-04-15 11:35:09,777 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=cosine, inertia=242.6761


/home/gabrielsanchez/repos/TFG-Chatbot/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2026-04-15 11:35:15,903 | INFO | tfg_validation_clustering | Fitting K-Means with k=6, distance=cosine
2026-04-15 11:35:16,056 | INFO | tfg_validation_clustering | K-Means converged at iteration 12
2026-04-15 11:35:16,057 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=cosine, inertia=52.2049


/home/gabrielsanchez/repos/TFG-Chatbot/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...

2026-04-15 11:35:27,974 | INFO | tfg_validation_clustering | Fitting K-Means with k=7, distance=cosine
2026-04-15 11:35:28,207 | INFO | tfg_validation_clustering | K-Means converged at iteration 26
2026-04-15 11:35:28,208 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=cosine, inertia=85.2500


/home/gabrielsanchez/repos/TFG-Chatbot/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2026-04-15 11:35:29,723 | INFO | tfg_validation_clustering | Fitting K-Means with k=7, distance=cosine
2026-04-15 11:35:29,942 | INFO | tfg_validation_clustering | K-Means converged at iteration 13
2026-04-15 11:35:29,943 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=cosine, inertia=626.0892


/home/gabrielsanchez/repos/TFG-Chatbot/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Figuras UMAP/PCA guardadas.


In [21]:
# Gráficas UMAP solo con clusters predichos (sin comparación con reales)
for _, row in df_best.iterrows():
    ds, rep = row["dataset"], row["representation"]
    X = representations[ds][rep]
    if X is None:
        continue

    model = GenericKMeans(
        n_clusters=int(row["k"]),
        algorithm=row["algorithm"],
        distance=row["distance"],
        random_state=RANDOM_SEED,
        m=2.0 if row["algorithm"] == "fcm" else None,
    )
    X_arr = X.toarray() if hasattr(X, "toarray") else X
    labels_pred = model.fit_predict(X_arr)

    X2 = reduce_to_2d(X, method="umap", random_state=RANDOM_SEED)

    fig, ax = plt.subplots(1, 1, figsize=(7, 5))
    ax.scatter(X2[:, 0], X2[:, 1], c=labels_pred, s=10, cmap="tab20")
    ax.set_title(f"UMAP/PCA por cluster predicho | {ds}-{rep}")
    
    save_figure(fig, f"umap_pred_only_{ds}_{rep}.png")
    plt.close(fig)

print("Figuras UMAP (solo predichas) guardadas.")

2026-04-15 11:40:13,069 | INFO | tfg_validation_clustering | Fitting K-Means with k=2, distance=cosine
2026-04-15 11:40:13,243 | INFO | tfg_validation_clustering | K-Means converged at iteration 12
2026-04-15 11:40:13,244 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=cosine, inertia=242.6761


/home/gabrielsanchez/repos/TFG-Chatbot/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2026-04-15 11:40:15,141 | INFO | tfg_validation_clustering | Fitting K-Means with k=6, distance=cosine
2026-04-15 11:40:15,275 | INFO | tfg_validation_clustering | K-Means converged at iteration 12
2026-04-15 11:40:15,276 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=cosine, inertia=52.2049


/home/gabrielsanchez/repos/TFG-Chatbot/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2026-04-15 11:40:16,867 | INFO | tfg_validation_clustering | Fitting K-Means with k=2, distance=cosine
2026-04-15 11:40:16,995 | INFO | tfg_validation_clustering | K-Means converged at iteration 9
2026-04-15 11:40:16,996 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=cosine, inertia=523.9666


...

2026-04-15 11:40:26,585 | INFO | tfg_validation_clustering | Fitting K-Means with k=7, distance=cosine
2026-04-15 11:40:26,865 | INFO | tfg_validation_clustering | K-Means converged at iteration 26
2026-04-15 11:40:26,865 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=cosine, inertia=85.2500


/home/gabrielsanchez/repos/TFG-Chatbot/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2026-04-15 11:40:28,285 | INFO | tfg_validation_clustering | Fitting K-Means with k=7, distance=cosine
2026-04-15 11:40:28,494 | INFO | tfg_validation_clustering | K-Means converged at iteration 13
2026-04-15 11:40:28,495 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=cosine, inertia=626.0892


/home/gabrielsanchez/repos/TFG-Chatbot/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Figuras UMAP (solo predichas) guardadas.


In [19]:
# t-SNE 2D para contraste cualitativo
for _, row in df_best_per_condition.iterrows():
    ds, rep = row["dataset"], row["representation"]
    X = representations[ds][rep]
    if X is None:
        continue

    X_arr = X.toarray() if hasattr(X, "toarray") else X
    n_samples = X_arr.shape[0]
    if n_samples < 50:
        continue

    m_val = row.get("m", 2.0)
    if pd.isna(m_val):
        m_val = 2.0 if row["algorithm"] == "fcm" else None
    
    model = GenericKMeans(
        n_clusters=int(row["k"]),
        algorithm=row["algorithm"],
        distance=row["distance"],
        random_state=RANDOM_SEED,
        m=m_val,
    )
    labels_pred = model.fit_predict(X_arr)

    perplexity = min(30, max(5, n_samples // 50))
    tsne = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=perplexity, init="pca")
    X2 = tsne.fit_transform(X_arr)

    labels_true = datasets[ds]["label"].astype(str).values
    _, true_ids = np.unique(labels_true, return_inverse=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(X2[:, 0], X2[:, 1], c=labels_pred, s=10, cmap="tab20")
    axes[0].set_title(f"t-SNE por cluster predicho | {ds}-{rep}")

    axes[1].scatter(X2[:, 0], X2[:, 1], c=true_ids, s=10, cmap="tab20")
    axes[1].set_title(f"t-SNE por etiqueta real | {ds}-{rep}")

    save_figure(fig, f"tsne_{ds}_{rep}.png")
    plt.close(fig)

print("Figuras t-SNE guardadas.")

2026-04-15 11:36:42,419 | INFO | tfg_validation_clustering | Fitting K-Means with k=2, distance=cosine
2026-04-15 11:36:42,589 | INFO | tfg_validation_clustering | K-Means converged at iteration 12
2026-04-15 11:36:42,589 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=cosine, inertia=242.6761
2026-04-15 11:36:51,212 | INFO | tfg_validation_clustering | Fitting K-Means with k=6, distance=cosine
2026-04-15 11:36:51,347 | INFO | tfg_validation_clustering | K-Means converged at iteration 12
2026-04-15 11:36:51,348 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=cosine, inertia=52.2049
2026-04-15 11:36:59,397 | INFO | tfg_validation_clustering | Fitting K-Means with k=2, distance=cosine
2026-04-15 11:36:59,531 | INFO | tfg_validation_clustering | K-Means converged at iteration 9
2026-04-15 11:36:59,531 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=cosine, inertia=523.9666
2026-0

In [20]:
# Incertidumbre difusa (solo FCM)
fuzzy_uncertainty_rows = []

for _, row in df_best_per_condition.iterrows():
    if row["algorithm"] != "fcm":
        continue

    ds, rep = row["dataset"], row["representation"]
    X = representations[ds][rep]
    if X is None:
        continue

    X_arr = X.toarray() if hasattr(X, "toarray") else X
    m_val = row.get("m", 2.0)
    if pd.isna(m_val):
        m_val = 2.0
    
    model = GenericKMeans(
        n_clusters=int(row["k"]),
        algorithm="fcm",
        distance=row["distance"],
        random_state=RANDOM_SEED,
        m=m_val,
    )
    model.fit(X_arr)
    U = model.membership_

    eps = 1e-12
    entropy = -np.sum(U * np.log(np.maximum(U, eps)), axis=1)
    max_membership = U.max(axis=1)

    fuzzy_uncertainty_rows.append({
        "dataset": ds,
        "representation": rep,
        "entropy_mean": float(np.mean(entropy)),
        "entropy_std": float(np.std(entropy)),
        "max_membership_mean": float(np.mean(max_membership)),
        "max_membership_std": float(np.std(max_membership)),
    })

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(entropy, bins=30, color="steelblue", alpha=0.8)
    axes[0].set_title(f"Entropia de membresia | {ds}-{rep}")

    axes[1].hist(max_membership, bins=30, color="darkorange", alpha=0.8)
    axes[1].set_title(f"Max membership | {ds}-{rep}")

    save_figure(fig, f"fuzzy_uncertainty_{ds}_{rep}.png")
    plt.close(fig)

df_fuzzy_uncertainty = pd.DataFrame(fuzzy_uncertainty_rows)
if not df_fuzzy_uncertainty.empty:
    save_table(df_fuzzy_uncertainty, "fuzzy_uncertainty_summary.csv")

df_fuzzy_uncertainty

""


## Discusión guiada

Completar tras ejecutar:
1. Algoritmo ganador por dataset y representación (según ASW y desempate por coste).
2. Consistencia de tendencias entre semillas y valores de k.
3. Trade-off calidad-coste entre variantes (ASW/CH frente a runtime y n_iter).
4. Diferencias por representación y por dataset (BoW, TF-IDF, Nomic).

Sugerencia: usar `summary_best_per_condition.csv`, `summary_global_by_variant.csv` y `summary_top_configs_by_asw.csv` como base del texto final.

Nota metodológica: en esta iteración no se realizan pruebas de significancia; las conclusiones son descriptivas y exploratorias.

## Amenazas a la validez

1. Sensibilidad a hiperparámetros (`k`, `m`, `max_features`, `min_df`, `perplexity`).
2. Sesgo por tamaño muestral en modo piloto y posibles truncamientos.
3. UMAP/t-SNE son técnicas estocásticas y dependientes de parámetros.
4. Coste computacional y disponibilidad de Nomic/Ollama pueden alterar comparabilidad práctica.
5. Etiquetas reales en algunos datasets pueden ser gruesas o parciales para validación externa.
6. No se realizan inferencias estadísticas en esta versión; los hallazgos deben interpretarse como evidencia descriptiva, no confirmatoria.

## Conclusiones y trabajo futuro

Plantilla para cierre:
1. Resumen de mejor configuración por dataset.
2. Balance rendimiento-calidad (ASW/CH/XB vs tiempo).
3. Recomendaciones metodológicas para el sistema TFG-Chatbot.
4. Siguientes pasos: HDBSCAN, robustez cross-domain, validación externa adicional (ARI/NMI si hay etiquetas consistentes), análisis de sensibilidad completo.

Nota operativa: tras validar piloto, desactivar `PILOT_MODE` para ejecución final completa sin alterar metodología.